In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.ml.feature import VectorAssembler

In [3]:
spark = SparkSession.builder \
  .appName("FeatureSelection_LR") \
  .getOrCreate()

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/18 22:05:01 INFO SparkEnv: Registering MapOutputTracker
25/12/18 22:05:01 INFO SparkEnv: Registering BlockManagerMaster
25/12/18 22:05:01 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
25/12/18 22:05:01 INFO SparkEnv: Registering OutputCommitCoordinator


In [4]:
df = spark.read.parquet("gs://jyairbnb/data_clean/joined_Q1")

After re-examining the linear relationships among the features, we found that several columns created during the calendar cleaning and feature-engineering steps were strongly linearly correlated. Therefore, we dropped those features.

Since my teammate will train a GBT model while I am using Linear Regression, I only need the indexed categorical features, not the one-hot encoded vectors. Therefore, I dropped all the vec columns.

In [5]:
cols_to_drop = [
    "days_available",
    "days_booked",
    "total_days",
    "available_rate",
    "price_per_booked_day",
    "neighbourhood_group_cleansed_vec",
    "property_type_vec",
    "room_type_vec",
    "host_is_superhost_vec",
    "instant_bookable_vec"
]

df = df.drop(*cols_to_drop)

In [6]:
df.write.mode("overwrite").parquet("gs://jyairbnb/data_clean/Final_LR/")

In [7]:
# Remove id and time columns because we don't need them as features for ML training.
exclude_cols = ["listing_id", "year", "week"]

feature_cols = [
    col for (col, dtype) in df.dtypes
    if dtype in ["int", "bigint", "double"]
    and col not in exclude_cols
]

# We removed avg_price and demand_score because one is the target variable and the other 
# is derived from the target, which would cause data leakage.
feature_cols = [c for c in feature_cols if c not in ["avg_price", "demand_score"]]

print("Number of features:", len(feature_cols))
print("Features:", feature_cols)

Number of features: 19
Features: ['booked_ratio', 'trend_yearly', 'bathrooms_clean', 'beds_clean', 'accommodates_clean', 'bedrooms_clean', 'amenities_count', 'host_total_listings_count_clean', 'number_of_reviews_clean', 'number_of_reviews_ltm_clean', 'review_scores_rating_clean', 'review_scores_cleanliness_clean', 'review_scores_accuracy_clean', 'review_scores_location_clean', 'neighbourhood_group_cleansed_index', 'property_type_index', 'room_type_index', 'host_is_superhost_index', 'instant_bookable_index']


## Correlation matrix

I use correlation-based filtering to keep only features that show a meaningful linear relationship with the target variable, so that weak or irrelevant features are removed before model training.

In [8]:
# Compute the correlation for each feature with the learning target(avg_price).

corr_list = []
for col in feature_cols:
    corr_value = df.stat.corr(col, "avg_price")
    corr_list.append((col, corr_value))

# Convert to DataFrame
df_corr = spark.createDataFrame(corr_list, ["feature", "corr_with_price"])

# Show top correlations
df_corr.orderBy(F.abs(F.col("corr_with_price")).desc()).show(20)

+--------------------+--------------------+
|             feature|     corr_with_price|
+--------------------+--------------------+
|  accommodates_clean| 0.10074780916756909|
|     bathrooms_clean| 0.08168159865709716|
|      bedrooms_clean|  0.0696479159896202|
|          beds_clean|0.056841547833511155|
|        booked_ratio|-0.04505346352124337|
|review_scores_rat...|0.026539363785240185|
| property_type_index|0.017799206551230635|
|     amenities_count|-0.01777927882546...|
|review_scores_acc...| 0.01775985999475728|
|     room_type_index|0.016160517497594187|
|number_of_reviews...|-0.00935459158532089|
|neighbourhood_gro...|0.008625574684874628|
|review_scores_cle...|0.008367210642333997|
|number_of_reviews...|0.008265856540579312|
|host_is_superhost...|0.005992615271061928|
|        trend_yearly|0.002300589558573...|
|review_scores_loc...|-0.00171937623954...|
|instant_bookable_...|0.001047907620229...|
|host_total_listin...|-0.00102483276596...|
+--------------------+----------

In [9]:
corr_threshold = 0.01   # Correlation Threshold

selected_by_corr = (
    df_corr
    .filter(F.abs(F.col("corr_with_price")) > corr_threshold)   # Filter by correlation strength
    .select("feature")
    .rdd.flatMap(lambda x: x)    # Convert to list
    .collect()
)
print("Features kept by correlation:", selected_by_corr)

Features kept by correlation: ['booked_ratio', 'bathrooms_clean', 'beds_clean', 'accommodates_clean', 'bedrooms_clean', 'amenities_count', 'review_scores_rating_clean', 'review_scores_accuracy_clean', 'property_type_index', 'room_type_index']


## Low variance filter

Because low variance features will not help the model. I remove features with very low variance to eliminate variables that contain little information and are unlikely to help the model learn meaningful patterns.

In [10]:
variance_list = []

# Compute variance for each feature
for col in feature_cols:
    var_value = df.agg(F.variance(F.col(col))).first()[0]
    variance_list.append((col, var_value))

# Create variance DataFrame
df_var = spark.createDataFrame(variance_list, ["feature", "variance"])

# Variance Threshold
variance_threshold = 0.01

selected_by_variance = (
    df_var.filter(F.col("variance") > variance_threshold)
          .select("feature")
          .rdd.flatMap(lambda x: x)    # Convert to list
          .collect()
)

print("Features kept by variance:", selected_by_variance)

Features kept by variance: ['booked_ratio', 'trend_yearly', 'bathrooms_clean', 'beds_clean', 'accommodates_clean', 'bedrooms_clean', 'amenities_count', 'host_total_listings_count_clean', 'number_of_reviews_clean', 'number_of_reviews_ltm_clean', 'review_scores_rating_clean', 'review_scores_cleanliness_clean', 'review_scores_accuracy_clean', 'review_scores_location_clean', 'neighbourhood_group_cleansed_index', 'property_type_index', 'room_type_index', 'host_is_superhost_index', 'instant_bookable_index']


## Final Selection

In [11]:
# Final selected features = intersection of both methods
final_features = list(set(selected_by_corr).intersection(set(selected_by_variance)))

print("Final selected features:", final_features)

Final selected features: ['review_scores_accuracy_clean', 'bathrooms_clean', 'amenities_count', 'booked_ratio', 'room_type_index', 'review_scores_rating_clean', 'beds_clean', 'accommodates_clean', 'property_type_index', 'bedrooms_clean']


In [12]:
# Create assembler for final selected feature list
assembler_final = VectorAssembler(
    inputCols=final_features,
    outputCol="features"
)

df_train = assembler_final.transform(df).select("features", "avg_price")
df_train = df_train.withColumnRenamed("avg_price", "price")
df_train.show(5)

+--------------------+-----+
|            features|price|
+--------------------+-----+
|[161.0,1.0,-1.0,0...|170.0|
|[161.0,1.5,1.0,0....| 80.0|
|[161.0,1.0,-1.0,0...|165.0|
|[161.0,1.0,1.0,0....|230.0|
|[161.0,1.5,1.0,1....|550.0|
+--------------------+-----+
only showing top 5 rows



In [13]:
df_train.write.mode("overwrite").parquet("gs://jyairbnb/lr_ready/")